In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# HF空间分配：冻结allocator的24项独立验证

复用现有3-cell运行骨架，从B开发分支获取已审fixed-LF方法版本。默认MODE='validation'，已填入fit-20260909-152025-689961的冻结allocator路径，无需手工修改。运行24项独立验证：96图、288评分路径；不refit、不挑换样本、不追加候选。此前fit完整但不构成成功证据。

输出保存到Drive的CEG-WM/development/survival-allocator-v1/validation-时间目录。需要Colab GPU及Secrets中的HF_TOKEN、CEG_WM_ROOT_KEY；环境版本仅记录。检查report.json、rows.jsonl及图像；失败行保留。

uniform、probe、survival共享量化LF参考，仅求HF幅度，总预算包含交叉项并检查实际dtype；不声称最终低频谱不变。uniform对survival是分配效果比较，original保持历史系统参考。仅clean、AWGN σ.02后clip、JPEG50；报告最终画质及正负分离，无真实科学结果或固定FPR结论。


In [ ]:
import json, os, pathlib, subprocess, sys, datetime
from importlib.metadata import version, PackageNotFoundError

REPO='https://github.com/RICHAAARC/CEG-WM.git'
BRANCH='dev/survival-allocator-v1'
EXPECTED_EXACT='aca1ed7fe2b179557b76efd68c6451ce77d3588e'
checkout=pathlib.Path('/content/survival-allocator-v1-github')
runtime_root=pathlib.Path('/content/ceg-method-models')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/development/survival-allocator-v1')
MODE='validation'  # fit / validation / plan
ALLOCATOR=pathlib.Path('/content/drive/MyDrive/CEG-WM/development/survival-allocator-v1/fit-20260909-152025-689961/allocator.json')
STAGE='validation' if MODE == 'validation' else 'fit'
output_root=drive_root/(STAGE+'-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
if not checkout.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin',BRANCH],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate','lpips','torchmetrics','scipy','sentencepiece'],check=True)
environment={}
for package in ('torch','diffusers','transformers','accelerate'):
    try: environment[package]=version(package)
    except PackageNotFoundError: environment[package]=None
print({'branch':BRANCH,'code':EXPECTED_EXACT,'versions':environment})
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
if MODE != 'plan':
    from google.colab import userdata
    child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
    child_env['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY') or ''
command=[sys.executable,'-m','experiments.run_survival_allocator_dev','--mode',MODE,
         '--roster',str(checkout/'configs/parallel_method_dev'/(STAGE+'.json')),
         '--output',str(output_root),'--runtime-root',str(runtime_root)]
if MODE == 'validation': command += ['--allocator',str(ALLOCATOR)]
completed=subprocess.run(command,cwd=checkout,env=child_env,check=False)
print('输出目录:', output_root)
if completed.returncode != 0:
    raise RuntimeError(f'开发入口退出码 {completed.returncode}；已写入的结果与失败行保留在 {output_root}')
report_path=output_root/'report.json'
if report_path.exists():
    report=json.loads(report_path.read_text())
    print({'report':str(report_path),'row_errors':report.get('row_errors'),'claim':report.get('claim')})
